# Godot DQN Agent

This notebook runs the training loop, connecting the `DQNAgent` to the `GodotEnv` websocket server.

In [ ]:
import asyncio
import websockets
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Import our modular DQN components
from dqn_agent import DQNAgent

class GodotEnv:
    def __init__(self, host='127.0.0.1', port=11000):
        self.host = host
        self.port = port
        self.server = None
        self.websocket = None
        self.last_state = []
        self.last_reward = 0.0
        self.last_done = False
        self.last_score = 0.0
        self.response_event = asyncio.Event()
        
    async def start_server(self):
        self.server = await websockets.serve(self.handler, self.host, self.port)
        print(f"WebSocket server started at ws://{self.host}:{self.port}. Waiting for Godot to connect...")
        
    async def handler(self, websocket, path):
        print("Godot connected!")
        self.websocket = websocket
        try:
            async for message in websocket:
                data = json.loads(message)
                self.last_state = data.get('state', [])
                self.last_reward = data.get('reward', 0.0)
                self.last_done = data.get('done', False)
                self.last_score = data.get('score', 0.0)
                self.response_event.set()
        except websockets.exceptions.ConnectionClosed:
            print("Godot disconnected.")
            self.websocket = None
            
    async def step(self, action: int):
        if self.websocket:
            self.response_event.clear()
            await self.websocket.send(json.dumps({"action": action}))
            await self.response_event.wait()
            return self.last_state, self.last_reward, self.last_done, self.last_score
        return [], 0.0, False, 0.0

    async def reset(self):
        if self.websocket:
            self.response_event.clear()
            await self.websocket.send(json.dumps({"command": "reset"}))
            await self.response_event.wait()
            return self.last_state
        return []


In [ ]:
def plot_metrics(episode_rewards, moving_avg, epsilons):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(episode_rewards, alpha=0.3, color='blue', label='Episode Reward')
    ax1.plot(moving_avg, color='red', label='Moving Avg (100 eps)')
    ax1.set_title('Training Rewards')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.legend()
    
    ax2.plot(epsilons, color='orange')
    ax2.set_title('Epsilon Decay')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Epsilon')
    
    plt.show()


In [ ]:
async def train():
    # Hyperparameters
    config = {
        "gamma": 0.99,
        "epsilon_start": 1.0,
        "epsilon_min": 0.05,
        "epsilon_decay": 0.995,
        "batch_size": 64,
        "learning_rate": 0.001,
        "buffer_size": 100000
    }
    
    state_size = 4
    action_size = 6
    agent = DQNAgent(state_size, action_size, config)
    env = GodotEnv()
    
    await env.start_server()
    
    # Wait for Godot to connect
    while env.websocket is None:
        await asyncio.sleep(1)
        
    num_episodes = 500
    target_update_freq = 10
    
    episode_rewards = []
    moving_avg = []
    epsilons = []
    
    for episode in range(num_episodes):
        state = await env.reset()
        total_reward = 0
        done = False
        step_count = 0
        
        while not done and env.websocket is not None:
            action = agent.get_action(state)
            next_state, reward, done, score = await env.step(action)
            
            agent.memory.add(state, action, reward, next_state, done)
            loss = agent.train_step()
            
            state = next_state
            total_reward += reward
            step_count += 1
            
            # Failsafe for getting stuck
            if step_count > 500:
                done = True
                
        agent.update_epsilon()
        if episode % target_update_freq == 0:
            agent.update_target_network()
            
        episode_rewards.append(total_reward)
        epsilons.append(agent.epsilon)
        
        # Calculate moving average
        if len(episode_rewards) >= 100:
            moving_avg.append(np.mean(episode_rewards[-100:]))
        else:
            moving_avg.append(np.mean(episode_rewards))
            
        if episode % 10 == 0:
            plot_metrics(episode_rewards, moving_avg, epsilons)
            print(f"Episode {episode}/{num_episodes} | Reward: {total_reward:.2f} | Epsilon: {agent.epsilon:.2f}")

# To run the training loop in Jupyter:
# asyncio.create_task(train())